In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# Function 1: Calculate ability estimates and its standard error for a given vector of binary responses
# and parameters of item difficulty and item discrimination. The resulting ability estimates typically
# range from -4 to +4 on a logistic scale (this is why I defined "ability_range" as -4 to +4). The higher
# values of ability indicate higher levels of achievement on the test.

# Standard error (SE) values range from 0 to positive infinity. The large the SE value, the higher the error. We
# typically want SE of 0.3 or less (if we want to let the test terminate itself earlier based on a precision
# level)

# Define constants for integration
QUADRATURE_POINTS = 41
ABILITY_RANGE = (-4, 4)

def calculate_eap_ability(responses, discrimination, difficulty):
    # Define quadrature points and weights for numerical integration
    abilities = np.linspace(ABILITY_RANGE[0], ABILITY_RANGE[1], QUADRATURE_POINTS)
    weights = norm.pdf(abilities) / np.sum(norm.pdf(abilities))

    # Initialize arrays for ability computation
    posterior_numerator = np.zeros(QUADRATURE_POINTS)

    # Calculate the posterior probability
    for i in range(len(responses)):
        prob_correct = 1 / (1 + np.exp(-discrimination[i] * (abilities - difficulty[i])))
        prob_incorrect = 1 - prob_correct

        # Update posterior probabilities
        posterior_numerator += (responses[i] * np.log(prob_correct) + (1 - responses[i]) * np.log(prob_incorrect))

    posterior_denominator = np.exp(posterior_numerator - np.max(posterior_numerator))  # Avoid overflow
    posterior_denominator /= posterior_denominator.sum()

    # Compute ability
    eap_ability = np.sum(abilities * posterior_denominator)

    # Compute the standard error of the ability estimate
    eap_variance = np.sum((abilities - eap_ability) ** 2 * posterior_denominator)
    eap_standard_error = np.sqrt(eap_variance)

    return eap_ability, eap_standard_error

In [ ]:
# Example usage:

# Load the sample dataset
data = pd.read_csv('sample_data.csv')

# Select three questions to estimate ability (I randomly selected three questions
# for this example)
selected_items = data[data['item_id'].isin([2053236, 2053237, 2053238])]

# Calculate EAP ability for a student
responses = selected_items['response'].values
discrimination = selected_items['discrimination'].values
difficulty = selected_items['difficulty'].values

example_eap_ability = calculate_eap_ability(responses, discrimination, difficulty)
print(f"EAP Ability Estimate: {example_eap_ability}")

EAP Ability Estimate: (-1.1100230853955617, 1.0516078368568322)


In [ ]:
# Function 2: Calculate Fisher information for a given ability level and parameters
# of item difficulty and item discrimination. Resulting Fisher information values are
# a float type numbers where the larger the value, the more informative a question.
def calculate_fisher_information(data, item_id, ability):
    # Filter the dataset for the specified item_id
    item_data = data[data['item_id'] == item_id]

    if item_data.empty:
        raise ValueError(f"Item ID {item_id} not found in the dataset.")

    # Extract difficulty and discrimination values for the item
    difficulty = item_data['difficulty'].values[0]
    discrimination = item_data['discrimination'].values[0]

    # Compute the probability of a correct response under the 2PL model
    prob_correct = 1 / (1 + np.exp(-discrimination * (ability - difficulty)))
    prob_incorrect = 1 - prob_correct

    # Fisher information for the specified item
    fisher_info = discrimination ** 2 * prob_correct * prob_incorrect

    return fisher_info

In [ ]:
# Example usage:

# Load the sample dataset
data = pd.read_csv('sample_data.csv')

# Calculate Fisher information for an example item and ability
example_fisher_info = calculate_fisher_information(data, item_id = 2053236, ability = -1)
print(f"Fisher Information: {example_fisher_info}")

Fisher Information: 0.484625927746153
